# Phase 1: Geometry Optimization and Harmonic Frequencies

**System:** CH3 + O(3P) -> HCHO + H (55 +/- 5%) / CO + H2 + H (45 +/- 5%)

This notebook optimizes geometries and computes harmonic vibrational
frequencies for every species on the CH3 + O potential energy surface, at
a DFT level of theory, using Psi4 (open source).

**What this notebook does NOT do:** CCSD(T) single-point refinement
(Phase 3), multireference diagnostics (Phase 2, done first on these
geometries before trusting CCSD(T)), or any kinetics. This is geometry
and thermochemistry input generation only.

**Species optimized here:**

| Species | Role | Spin multiplicity | Charge |
|---|---|---|---|
| CH3 | reactant | 2 (doublet) | 0 |
| O | reactant | 3 (triplet, 3P ground state) | 0 |
| CH3O | intermediate (methoxy) | 2 (doublet) | 0 |
| CH2OH | intermediate (methoxy isomer) | 2 (doublet) | 0 |
| HCHO | product, channel 1 | 1 (singlet) | 0 |
| H | product | 2 (doublet) | 0 |
| HCO | likely intermediate en route to channel 2 | 2 (doublet) | 0 |
| H2 | product, channel 2 | 1 (singlet) | 0 |
| CO | product, channel 2 | 1 (singlet) | 0 |

Starting geometries below are reasonable chemical estimates (typical bond
lengths/angles), **not** literature-precise coordinates -- refining them is
the entire point of running the optimization. What matters and is not up
for adjustment: the charge and spin multiplicity of each species, which are
physically fixed.

In [ ]:
import psi4
import numpy as np
import pandas as pd
import os

# Output directory for raw Psi4 output files (kept out of git via .gitignore
# pattern on *.out, or explicitly copied into data/ if we want them tracked)
os.makedirs('../data/phase1_geometries', exist_ok=True)

psi4.set_memory('4 GB')
psi4.set_num_threads(4)

# Level of theory for Phase 1. wB97X-D chosen for a reasonable balance of
# accuracy and cost for geometry/frequency work; this is a placeholder
# choice to be revisited (see project_plan.md, Open Decisions) against what
# is defensible relative to dissertation-era method choices.
DFT_METHOD = 'wb97x-d'
BASIS = 'cc-pVTZ'

## Species definitions

Each molecule string below uses Psi4's Z-matrix-style input. The first
line of each string is `charge multiplicity`.

In [ ]:
species = {}

# --- Reactants ---

species['CH3'] = """
0 2
C
H 1 1.079
H 1 1.079 2 120.0
H 1 1.079 2 120.0 3 180.0
"""
# Planar D3h methyl radical, C-H ~1.079 Angstrom (typical literature value).

species['O'] = """
0 3
O
"""
# Ground-state atomic oxygen, triplet (3P).

# --- Intermediates ---

species['CH3O'] = """
0 2
O
C 1 1.39
H 2 1.10 1 108.0
H 2 1.10 1 108.0 3 120.0
H 2 1.10 1 108.0 3 -120.0
"""
# Methoxy radical, pyramidal at C, rough starting geometry.

species['CH2OH'] = """
0 2
C
O 1 1.40
H 2 0.97 1 108.0
H 1 1.09 2 110.0 3 120.0
H 1 1.09 2 110.0 3 -120.0
"""
# Hydroxymethyl radical (methoxy isomer), rough starting geometry.

species['HCO'] = """
0 2
C
O 1 1.18
H 1 1.12 2 124.0
"""
# Formyl radical, bent, rough starting geometry.

# --- Products ---

species['HCHO'] = """
0 1
C
O 1 1.20
H 1 1.10 2 121.0
H 1 1.10 2 121.0 3 180.0
"""
# Formaldehyde, planar C2v, rough starting geometry.

species['H'] = """
0 2
H
"""

species['H2'] = """
0 1
H
H 1 0.74
"""

species['CO'] = """
0 1
C
O 1 1.13
"""

print(f"{len(species)} species defined: {list(species.keys())}")

## Optimization and frequency loop

For each species: build the Psi4 molecule, optimize the geometry, then run
a harmonic frequency calculation at the optimized geometry. Results
(energy, optimized geometry, frequencies, zero-point energy) are collected
into a summary table and written to `data/` for use in later phases.

**Known risk to check for, not yet resolved:** open-shell species (CH3,
CH3O, CH2OH, HCO, H, O) need `reference uhf` or `reference uks` set
explicitly for DFT; closed-shell species (HCHO, H2, CO) use the default
restricted reference. This is handled per-species below, but the resulting
wavefunction stability should be checked (e.g., no negative frequencies
for a true minimum, and `<S^2>` close to the expected value for open-shell
cases) once this notebook actually runs -- not assumed to be fine.

In [ ]:
# Species requiring an unrestricted reference (open-shell: doublet or triplet)
OPEN_SHELL = {'CH3', 'O', 'CH3O', 'CH2OH', 'HCO', 'H'}

results = {}

for name, geom_string in species.items():
    print(f"\n{'='*50}\nProcessing {name}\n{'='*50}")

    psi4.core.set_output_file(f'../data/phase1_geometries/{name}_opt_freq.out', False)

    mol = psi4.geometry(geom_string)

    if name in OPEN_SHELL:
        psi4.set_options({'reference': 'uks'})
    else:
        psi4.set_options({'reference': 'rks'})

    # Single atoms (O, H) don't need geometry optimization -- Psi4 will
    # error or trivially converge; skip optimization for these and go
    # straight to a single-point + frequency call (frequencies will be
    # trivially empty for a single atom, but the energy is what matters).
    is_atom = name in ('O', 'H')

    try:
        if not is_atom:
            opt_energy = psi4.optimize(f'{DFT_METHOD}/{BASIS}', molecule=mol)
        else:
            opt_energy = psi4.energy(f'{DFT_METHOD}/{BASIS}', molecule=mol)

        freq_energy, wfn = psi4.frequencies(
            f'{DFT_METHOD}/{BASIS}', molecule=mol, return_wfn=True
        )

        frequencies = wfn.frequencies().to_array() if not is_atom else np.array([])
        n_imaginary = int(np.sum(frequencies < 0)) if len(frequencies) else 0

        results[name] = {
            'electronic_energy_Eh': opt_energy,
            'n_imaginary_freq': n_imaginary,
            'frequencies_cm-1': frequencies.tolist(),
            'geometry_final': mol.save_string_xyz(),
        }

        flag = 'OK (minimum)' if n_imaginary == 0 else f'CHECK: {n_imaginary} imaginary freq(s)'
        print(f"{name}: E = {opt_energy:.6f} Eh, {flag}")

    except Exception as e:
        print(f"FAILED for {name}: {e}")
        results[name] = {'error': str(e)}

    psi4.core.clean()

## Summary table

Every species should show `n_imaginary_freq = 0` to confirm it is a true
minimum on the potential energy surface. Any species with imaginary
frequencies needs its starting geometry reconsidered before moving on --
this is not something to paper over by just continuing to Phase 2.

In [ ]:
summary_rows = []
for name, r in results.items():
    if 'error' in r:
        summary_rows.append({'species': name, 'status': 'FAILED', 'energy_Eh': None, 'n_imaginary': None})
    else:
        summary_rows.append({
            'species': name,
            'status': 'OK' if r['n_imaginary_freq'] == 0 else 'CHECK GEOMETRY',
            'energy_Eh': r['electronic_energy_Eh'],
            'n_imaginary': r['n_imaginary_freq'],
        })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv('../data/phase1_geometries/phase1_summary.csv', index=False)
summary_df

## Next steps

1. Visually inspect optimized geometries (e.g., via a molecular viewer) to
   confirm they are chemically sensible, not just numerically converged.
2. Proceed to `notes/multireference_diagnostic_check.md` -- run T1
   diagnostics on these optimized geometries before trusting CCSD(T) for
   Phase 3 single-point energies.
3. If any species shows imaginary frequencies, revisit its starting
   geometry and re-optimize before proceeding.